In [1]:
import os

import pandas as pd
from hicona import HiconaCooler

In [2]:
FILE_FOLDER = "test_files"
FILE_PATHS = ["Rao_2014_IMR90_MboI_4DNFIJTOIGOI.mcool"] #"Rao_2014_HUVEC_MboI_4DNFIRMZ7QTE.mcool", 
RESOLUTION = 10_000
DISTANCE = 200_000_000
TOP_N = 500_000

In [3]:
def get_chroms_as_csvs(file_folder, file_path, resolution, distance, top_n):
    """Save chromosome level tables (and bin annotation) from an HiconaCooler object to csv."""
    
    handle = HiconaCooler(os.path.join(file_folder, file_path) + "::resolutions/" + str(resolution))
    out_dir = os.path.join(file_folder, "_".join(file_path.split("_")[0:3]) + f"_top_{top_n}")
    os.makedirs(out_dir)
    for tab, info in handle.tables(dist_thr=distance):
        tab.nsmallest(top_n, "spar_alpha").to_csv(os.path.join(out_dir, info["chromosome"] + ".csv"))
    handle.bins()[:].to_csv(os.path.join(out_dir, "bin_annotation.csv"))
    

In [4]:
def get_full_chroms_as_csvs(file_folder, file_path, resolution, distance):
    """Save chromosome level tables (and bin annotation) from an HiconaCooler object to csv."""
    
    handle = HiconaCooler(os.path.join(file_folder, file_path) + "::resolutions/" + str(resolution))
    out_dir = os.path.join(file_folder, "_".join(file_path.split("_")[0:3]) + f"_full")
    os.makedirs(out_dir)
    for tab, info in handle.tables(dist_thr=distance):
        tab.to_csv(os.path.join(out_dir, info["chromosome"] + ".csv"))
    handle.bins()[:].to_csv(os.path.join(out_dir, "bin_annotation.csv"))
    

In [5]:
def get_joint_top_n(file_folder, top_n):
    """Merge individual chromosome tables to extract the top n overall (and save it)."""
    
    file_paths = [os.path.join(file_folder, f) for f in os.listdir(file_folder) if f.startswith("chr")]
    datafs = [pd.read_csv(f, index_col=0) for f in file_paths]
    final_df = pd.concat(datafs)
    final_df.nsmallest(top_n, "spar_alpha").to_csv(os.path.join(file_folder, "chrall.csv"))
    

In [3]:
def get_full_joint(file_folder):
    """Merge individual chromosome tables to extract the top n overall (and save it)."""
    
    file_paths = [os.path.join(file_folder, f) for f in os.listdir(file_folder) if f.startswith("chr")]
    datafs = [pd.read_csv(f, index_col=0) for f in file_paths]
    final_df = pd.concat(datafs)
    final_df.to_csv(os.path.join(file_folder, "chrall.csv"))
    

In [15]:
def get_joint_top_alpha(file_folder, alpha_val):
    """Merge individual chromosome tables and extract the top overall by alpha (and save it)."""
    
    file_paths = [os.path.join(file_folder, f) for f in os.listdir(file_folder) if f.startswith("chr")]
    datafs = [pd.read_csv(f, index_col=0) for f in file_paths]
    final_df = pd.concat(datafs)
    final_df = final_df[final_df["spar_alpha"] < alpha_val]
    final_df.to_csv(os.path.join(file_folder, "chrall_alpha.csv"))
    

Actually run the functions

In [7]:
for file_path in FILE_PATHS:
    get_chroms_as_csvs(FILE_FOLDER, file_path, RESOLUTION, DISTANCE, TOP_N)

In [8]:
for file_path in FILE_PATHS:
    get_full_chroms_as_csvs(FILE_FOLDER, file_path, RESOLUTION, DISTANCE)

In [9]:
for file_path in FILE_PATHS:
    out_dir = os.path.join(FILE_FOLDER, "_".join(file_path.split("_")[0:3]) + f"_top_{TOP_N}")
    get_joint_top_n(out_dir, TOP_N)

In [ ]:
for file_path in FILE_PATHS:
    out_dir = os.path.join(FILE_FOLDER, "_".join(file_path.split("_")[0:3]) + f"_full")
    get_full_joint(out_dir)

In [16]:
for file_path in FILE_PATHS:
    out_dir = os.path.join(FILE_FOLDER, "_".join(file_path.split("_")[0:3]))
    get_joint_top_alpha(out_dir, 0.05)

In [17]:
test_df = pd.read_csv("test_files/Rao_2014_HUVEC/chrall_alpha.csv", index_col = 0)

In [18]:
test_df

,bin1_id,bin2_id,count,exp_ratio,spar_alpha
1566168,226259,227260,1,1.000000,0.0204
1245912,225208,225233,1,0.415037,0.0232
1469577,225939,226549,1,1.000000,0.0243
1245913,225208,225237,3,1.321928,0.0249
1028638,224499,224583,4,2.321928,0.0253
...,...,...,...,...,...
851452,242626,243249,1,1.000000,0.0499
996348,243509,248115,2,1.584963,0.0499
1025539,244689,248398,2,1.584963,0.0499
1029907,244690,245753,2,1.584963,0.0499


Remake annotation file separately